# Marketing & Sales Analytics - Business Analytics and Visualization (202046719)
**Problem:** Did our marketing campaign increase sales, and which regions / segments / categories should we focus on?

| Section | Syllabus Unit | Related Lab |
|---|---|---|
| 1. Descriptive analytics | Unit 1 | Labs 1-4 |
| 2. Sampling & estimation | Unit 2 | Labs 5-6 |
| 3. Hypothesis testing | Unit 3 | Lab 11 |
| 4. ANOVA | Unit 4 | - |
| 5. Data visualization | Unit 5 | Lab 10 |
| 6. Regression & forecasting | Unit 6 | Lab 7 |
| 7. Case study conclusions | Unit 7 | - |

All analysis functions live in `analytics.py`, which is also used by the dashboard (`dashboard.py`). The last cell launches the dashboard from this notebook.

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
import analytics as an

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

# Put your own Kaggle CSV named superstore.csv here (columns as in analytics.py) - otherwise synthetic data is used
df = an.load_data("superstore.csv")
df.head()

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
print(df.shape)
df.info()
df.describe()

## 1. Descriptive Analytics (Unit 1)
What happened? KPIs and breakdowns.

In [ ]:
for k, v in an.kpis(df).items():
    print(f"{k:18s}: {v:,.2f}")

In [ ]:
for col in ["region", "segment", "category"]:
    display(an.summary_by(df, col))

In [ ]:
# Classify orders as Average / Good / Excellent (like Lab 4) using quantiles of sales
df["order_class"] = pd.qcut(df["sales"], [0, .5, .85, 1], labels=["Average", "Good", "Excellent"])
df["order_class"].value_counts()

## 2. Sampling & Estimation (Unit 2)
Central Limit Theorem, confidence intervals, sample size, MLE.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["sales"], bins=40, ax=ax[0]).set_title("Population (skewed/irregular)")
for i, n in enumerate([5, 50]):
    sns.histplot(an.sample_means(df, "sales", n), bins=30, kde=True, ax=ax[i+1]).set_title(f"Sample means, n={n}")
plt.tight_layout(); plt.show()

In [ ]:
sample = df["sales"].sample(50, random_state=7)
m, lo, hi = an.confidence_interval(sample)
print(f"Sample mean = {m:.2f}, 95% CI = ({lo:.2f}, {hi:.2f}); true population mean = {df['sales'].mean():.2f}")
print("Sample size needed for +/-5 margin at 95% confidence:", an.required_sample_size(df['sales'].std(), 5))
mu, sigma = an.mle_normal(df["sales"])
print(f"MLE estimates -> mean = {mu:.2f}, std = {sigma:.2f}")

## 3. Hypothesis Testing (Unit 3 / Lab 11)
**H0:** mean sales (Campaign) = mean sales (Control)  
**H1:** mean sales (Campaign) > mean sales (Control)  
Significance level alpha = 0.05

In [ ]:
res = an.campaign_test(df)
pd.Series(res)

In [ ]:
sns.boxplot(data=df, x="group", y="sales"); plt.title("Sales: Campaign vs Control"); plt.show()
d = res["cohens_d"]
print("Effect size:", "small" if abs(d) < .5 else "medium" if abs(d) < .8 else "large", f"(d = {d:.2f})")
print("Power of the test:", round(an.power_of_test(abs(d), df['group'].value_counts().min()), 3))

In [ ]:
# One-sample Z-test (known sigma): is average order value different from 350?
z, p = an.z_test_mean(df["sales"], 350, df["sales"].std())
print(f"Z = {z:.3f}, p = {p:.4g}")

**Interpretation:** If p < 0.05 we reject H0 -> the campaign has a statistically significant positive effect. Type I error = wrongly rejecting a true H0 (probability alpha); Type II error = missing a real effect (1 - power).

## 4. Analysis of Variance (Unit 4)
Does average sales differ across groups? Why ANOVA instead of many t-tests? Multiple t-tests inflate the Type I error.

In [ ]:
for col in ["region", "segment", "category"]:
    f, p = an.anova_oneway(df, col)
    print(f"One-way ANOVA on {col:9s}: F = {f:8.3f}, p = {p:.4g}")

In [ ]:
an.anova_twoway(df, "region", "segment")

## 5. Data Visualization (Unit 5 / Lab 10)
Each chart type is chosen for the question it answers.

In [ ]:
monthly = df.groupby("month")["sales"].sum().reset_index()
px.line(monthly, x="month", y="sales", title="Trend over time -> line chart")

In [ ]:
px.bar(an.summary_by(df, "category").reset_index(), x="category", y="sales", color="category",
       title="Comparison across categories -> bar chart")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(df.pivot_table(index="region", columns="segment", values="sales", aggfunc="mean"),
            annot=True, fmt=".0f", cmap="YlGnBu", ax=ax[0]).set_title("Two-variable comparison -> heatmap")
sns.heatmap(df[["sales", "profit", "ad_spend", "discount"]].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", ax=ax[1]).set_title("Relationships -> correlation heatmap")
plt.tight_layout(); plt.show()

In [ ]:
px.scatter(df.sample(800, random_state=1), x="ad_spend", y="sales", color="group", trendline="ols",
           title="Relationship -> scatter plot with trendline")

## 6. Advanced Topics (Unit 6): Multiple Linear Regression & Forecasting

In [ ]:
model = an.regression(df)
print(model.summary())

In [ ]:
print(f"R-squared = {model.rsquared:.3f}")
plt.scatter(df["sales"], model.fittedvalues, alpha=.3); plt.plot([0, df.sales.max()], [0, df.sales.max()], "r--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Actual vs Predicted sales"); plt.show()

In [ ]:
ts, fc = an.forecast_sales(df, 6)
plt.figure(figsize=(10, 4)); plt.plot(ts, label="Actual"); plt.plot(fc, "--o", label="Forecast (Holt-Winters)")
plt.legend(); plt.title("6-month sales forecast"); plt.show()
fc.round(0)

## 7. Case Study Conclusions (Unit 7)
Write your findings here after running the notebook with your final data, for example:
- **Campaign effect:** (significant / not significant, effect size)
- **Best region / segment / category:** (from the ANOVA and heatmaps)
- **Drivers of sales:** (ad spend vs discount, from regression coefficients)
- **Outlook:** (from the forecast)
- **Recommendation:** (where to invest and what to avoid)

## Launch the Dashboard
This runs `dashboard.py`, which uses the same `analytics.py` as this notebook. It opens at http://localhost:8501. To stop it, run the second cell.

In [ ]:
import subprocess, sys
dash = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "dashboard.py"])
print("Dashboard running at http://localhost:8501")

In [ ]:
dash.terminate()   # stop the dashboard